# Notebook 4C — Visual recovery walkthrough for 12 focused study areas

This notebook is a deliberately small bridge from Notebook 3 to systematic recovery-family analysis. It uses actual VNP46A2 data for two spatial supports:

1. the reliability-qualified GHSL G3 pixels across each city/municipality; and
2. one 5×5 VIIRS kernel centred on the brightest fixed-baseline G3 pixel within that boundary.

The study set is restricted to the six initial locations—Tacloban, Ormoc, Baybay, Catbalogan, Borongan and Guiuan—plus Palo, Tanauan, Tolosa, Dulag, Basey and Lawaan. The latter six extend the overlap with the seven-municipality DEval proof of concept, in which Tacloban is already represented.

The notebook prioritises visible intermediate steps:

**boundary and support → quantile filtering → baseline → spatial change → temporal recovery → T50/T80/T90 → functional summary → trajectory grouping → mapped families**

The family labels are descriptive summaries of numerical similarity. They do not explain causation. Physical EO and survey evidence are reserved for validating and explaining the NTL families after they are established.

## Analytical choices retained from Notebook 3

- Direct `DNB_BRDF_Corrected_NTL` is the main signal; MQF = 0 defines a fresh/high-quality observation.
- GHSL G3 restricts the support to settlement classes 22, 23 and 30.
- Daily radiance is capped at the spatial 95th percentile. This is a default robustness filter for unusually bright pixels, not deletion of the underlying observations.
- Four-day non-overlapping medians and the 60-day pixel-median baseline are retained from Notebook 3.
- Recovery is calculated only from pixels observed in both the current composite and baseline.
- A value is withheld when spatial completeness is below 10%.
- All main figures use a 16:9 canvas (`1280 × 720`), `plotly_white`, transparent paper background and the same dashed blue Haiyan marker.

The DEval study provides a later validation route through very-high-resolution physical EO proxies and household/barangay/municipal survey evidence. It does not define the NTL processing choices used here.

In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import json
import re
import unicodedata
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import Point

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, pairwise_distances, silhouette_score

from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# ============================================================
# 2. PATHS, FOCUSED STUDY AREAS, AND SETTINGS
# ============================================================

PROJECT_DIR_OVERRIDE = Path(
    "/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/"
    "02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery"
)
project_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, PROJECT_DIR_OVERRIDE]
PROJECT_DIR = next((path for path in project_candidates if (path / "datasets").exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Set PROJECT_DIR_OVERRIDE to the project root containing datasets.")

DATA_DIR = PROJECT_DIR / "datasets"
VNP46_DIR = DATA_DIR / "VNP46"
PROCESSED_DIR = VNP46_DIR / "processed"
A2_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"

def find_dataset(patterns, label):
    matches = []
    for pattern in patterns:
        matches.extend(DATA_DIR.glob(pattern))
    matches = sorted({path.resolve() for path in matches})
    if not matches:
        raise FileNotFoundError(f"No {label} matched under {DATA_DIR}: {patterns}")
    if len(matches) > 1:
        print(f"{label}: multiple matches; using {matches[0]}")
    return matches[0]

GHSL_PATH = find_dataset(
    ["VNP46/GHSL_SMOD_E2015.tif", "ghsl/GHSL_SMOD_E2015.tif", "**/*GHSL*SMOD*.tif"],
    "GHSL SMOD raster",
)
MUNICIPALITIES_PATH = find_dataset(
    ["**/*MuniCities*.shp", "**/*municities*.shp", "**/*Muni*Cit*.shp", "**/*Municipal*.shp"],
    "MuniCities shapefile",
)

OUTPUT_DIR = PROJECT_DIR / "output" / "focused_recovery_walkthrough"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
for directory in (OUTPUT_DIR, FIGURE_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

STUDY_AREAS = [
    "Tacloban", "Ormoc", "Baybay", "Catbalogan", "Borongan", "Guiuan",
    "Palo", "Tanauan", "Tolosa", "Dulag", "Basey", "Lawaan",
]

EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_DAYS = 60
ANALYSIS_START = EVENT_DATE - pd.Timedelta(days=180)
PROFILE_END = EVENT_DATE + pd.Timedelta(days=363)
BASELINE_START = EVENT_DATE - pd.Timedelta(days=BASELINE_DAYS)
PRE_EVENT_END = EVENT_DATE - pd.Timedelta(days=1)

DNB_BAND = "DNB_BRDF_Corrected_NTL"
GAP_FILLED_BAND = "Gap_Filled_DNB_BRDF_Corrected_NTL"
MQF_BAND = "Mandatory_Quality_Flag"
SPATIAL_DIMS = ("y", "x")
GHSL_MASKS = {"G2": (23, 30), "G3": (22, 23, 30), "G4": (21, 22, 23, 30)}
SETTLEMENT_MASK = "G3"
SPATIAL_COMPLETENESS_PCT = 10.0
RQ_CLIP_PERCENTILE = 95.0
MAP_DISPLAY_QUANTILE = 0.98
MIN_BASELINE_OBSERVATIONS = {4: 3}
PERSISTENCE_BLOCKS = 2
MIN_EVENT_RETENTION_PCT = 20.0
MIN_EVENT_COMPOSITES = 8
MAX_INTERPRETABLE_GAP_DAYS = 24
KNOWN_FILL_VALUES = (-9999.0, -32768.0, 6553.5, 65535.0)

AGGREGATION_DAYS = 4
KERNEL_SIZE = 5
CLUSTER_HORIZON_DAYS = 180
CLUSTER_BIN_DAYS = 20
MIN_BIN_COMPOSITES = 2
CLUSTER_RANDOM_STATE = 42

print("Project:", PROJECT_DIR)
print("Municipality boundaries:", MUNICIPALITIES_PATH)
print("Focused study areas:", ", ".join(STUDY_AREAS))

Project: /Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery
Municipality boundaries: /Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/datasets/boundaries/MuniCities/MuniCities.shp
Focused study areas: Tacloban, Ormoc, Baybay, Catbalogan, Borongan, Guiuan, Palo, Tanauan, Tolosa, Dulag, Basey, Lawaan


In [3]:
municipalities = gpd.read_file(MUNICIPALITIES_PATH)

if municipalities.crs is None:
    raise ValueError("Municipality shapefile does not contain a CRS.")

municipalities = municipalities.loc[
    municipalities.geometry.notna()
    & ~municipalities.geometry.is_empty
].copy()

municipality_column = next(
    (
        column
        for column in (
            "unit_name", "NAME_2", "ADM2_EN", "MUNICITY",
            "MuniCity", "MUNICIPAL", "Municipali",
        )
        if column in municipalities.columns
    ),
    None,
)
province_column = next(
    (
        column
        for column in (
            "province_name", "NAME_1", "ADM1_EN",
            "PROVINCE", "Province",
        )
        if column in municipalities.columns
    ),
    None,
)

if municipality_column is None or province_column is None:
    raise KeyError(
        "Identify the municipality and province columns from: "
        f"{municipalities.columns.tolist()}"
    )

municipalities["unit_name"] = (
    municipalities[municipality_column].astype("string").fillna("").str.strip()
)
municipalities["province_name"] = (
    municipalities[province_column].astype("string").fillna("").str.strip()
)

municipality_names = (
    municipalities["unit_name"]
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("ascii")
    .str.casefold()
    .str.replace(r"\bcity\s+of\b|\bcity\b", "", regex=True)
    .str.replace(r"[^a-z0-9]+", " ", regex=True)
    .str.strip()
)

municipalities["study_name"] = municipality_names.map(
    {name.casefold(): name for name in STUDY_AREAS}
)

focused_municipalities = municipalities.loc[
    municipalities["study_name"].notna()
].copy()

In [4]:
# Copied from Notebook 3, source cell index 5
# ============================================================
# 4. VNP46A2 HELPERS
# ============================================================


def open_zarr_safely(path):
    try:
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks="auto",
            mask_and_scale=True,
            decode_cf=True,
        )
    except (ImportError, ModuleNotFoundError, ValueError):
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks=None,
            mask_and_scale=True,
            decode_cf=True,
        )


def standardise_date_dimension(ds):
    if "date" not in ds.variables:
        raise KeyError(f"No `date` variable found. Variables: {list(ds.variables)}")

    if "date" not in ds.coords:
        ds = ds.set_coords("date")

    observation_dim = ds["date"].dims[0]
    dates = pd.DatetimeIndex(pd.to_datetime(ds["date"].values)).normalize()
    ds = ds.assign_coords(date=(observation_dim, dates.values))

    if observation_dim != "date":
        ds = ds.swap_dims({observation_dim: "date"})

    return ds.sortby("date")


def prepare_spatial_metadata(ds):
    ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)

    if ds.rio.crs is None and "spatial_ref" in ds.variables:
        attrs = ds["spatial_ref"].attrs
        stored_crs = attrs.get("crs_wkt") or attrs.get("spatial_ref")

        if stored_crs is not None:
            ds = ds.rio.write_crs(stored_crs, inplace=False)

    if ds.rio.crs is None:
        x_min = float(ds["x"].min())
        x_max = float(ds["x"].max())
        y_min = float(ds["y"].min())
        y_max = float(ds["y"].max())

        if (
            -180 <= x_min <= 180
            and -180 <= x_max <= 180
            and -90 <= y_min <= 90
            and -90 <= y_max <= 90
        ):
            ds = ds.rio.write_crs("EPSG:4326", inplace=False)
        else:
            raise ValueError("The VNP46A2 CRS could not be recovered.")

    return ds


def clean_radiance(values):
    cleaned = values.astype("float32").where(np.isfinite(values))
    fill_values = list(KNOWN_FILL_VALUES)

    for source in (values.attrs, values.encoding):
        for key in ("_FillValue", "missing_value"):
            if source.get(key) is not None:
                fill_values.append(source[key])

    for fill_value in fill_values:
        try:
            fill_value = float(fill_value)
            if np.isfinite(fill_value):
                cleaned = cleaned.where(~np.isclose(cleaned, fill_value))
        except (TypeError, ValueError):
            continue

    return cleaned.where(cleaned >= 0)


if not A2_ZARR_PATH.exists():
    raise FileNotFoundError(f"VNP46A2 Zarr not found:\n{A2_ZARR_PATH}")

a2 = prepare_spatial_metadata(
    standardise_date_dimension(open_zarr_safely(A2_ZARR_PATH))
).sel(date=slice(ANALYSIS_START, PROFILE_END))

required_bands = [DNB_BAND, MQF_BAND]
missing_bands = [band for band in required_bands if band not in a2.data_vars]

if missing_bands:
    raise KeyError(
        f"Missing A2 bands: {missing_bands}\n"
        f"Available: {list(a2.data_vars)}"
    )

dnb = clean_radiance(a2[DNB_BAND])
mqf = a2[MQF_BAND]
dnb, mqf = xr.align(dnb, mqf, join="inner")

gap_filled = (
    clean_radiance(a2[GAP_FILLED_BAND])
    if GAP_FILLED_BAND in a2.data_vars
    else None
)

print("A2 dimensions:", dict(a2.sizes))
print("A2 CRS:", a2.rio.crs)
print("Available dates:", pd.Timestamp(a2.date.min().item()).date(), "to", pd.Timestamp(a2.date.max().item()).date())

KeyboardInterrupt: 

In [ ]:
study_area_provinces = {
    "Tacloban": "Leyte",
    "Ormoc": "Leyte",
    "Baybay": "Leyte",
    "Catbalogan": "Samar",
    "Borongan": "Eastern Samar",
    "Guiuan": "Eastern Samar",
    "Palo": "Leyte",
    "Tanauan": "Leyte",
    "Tolosa": "Leyte",
    "Dulag": "Leyte",
    "Basey": "Samar",
    "Lawaan": "Eastern Samar",
}

focused_municipalities = focused_municipalities.loc[
    focused_municipalities.apply(
        lambda row: (
            row["study_name"] in study_area_provinces
            and row["province_name"].strip().casefold()
            == study_area_provinces[row["study_name"]].casefold()
        ),
        axis=1,
    )
].copy()

duplicates = focused_municipalities["study_name"].value_counts()

if (duplicates > 1).any():
    display(
        focused_municipalities.loc[
            focused_municipalities["study_name"].isin(
                duplicates[duplicates > 1].index
            ),
            ["study_name", "unit_name", "province_name"],
        ]
    )

    raise ValueError(
        "Duplicate municipality–province matches remain. "
        "Inspect the displayed records."
    )

focused_municipalities["unit_name"] = focused_municipalities["study_name"]
focused_municipalities["unit_label"] = focused_municipalities["unit_name"]

In [ ]:
ROADS_PATH = find_dataset(
    [
        "**/*Roads*.shp",
        "**/*roads*.shp",
        "**/*Road*.shp",
    ],
    "Roads shapefile",
)


HAIYAN_TRACK_PATH = find_dataset(
    [
        "yolanda-path-line-/Yolanda Path Line.shp",
        "**/Yolanda Path Line.shp",
        "**/*Yolanda*Path*.shp",
        "**/*Haiyan*Path*.shp",
    ],
    "Haiyan/Yolanda path shapefile",
)

In [ ]:
roads = gpd.read_file(ROADS_PATH)

if roads.crs is None:
    raise ValueError("Roads shapefile does not contain a CRS.")

roads = roads.loc[roads.geometry.notna() & ~roads.geometry.is_empty].copy()

haiyan_track = gpd.read_file(HAIYAN_TRACK_PATH)
if haiyan_track.crs is None:
    raise ValueError("The Haiyan/Yolanda path shapefile does not contain a CRS.")
haiyan_track = haiyan_track.loc[
    haiyan_track.geometry.notna() & ~haiyan_track.geometry.is_empty
].copy()

In [ ]:
roads_raster_crs = roads.to_crs(a2.rio.crs)
haiyan_track_raster_crs = haiyan_track.to_crs(a2.rio.crs)

In [ ]:
# ============================================================
# MAP-PLOTTING HELPERS
# ============================================================

def degree_minute_label(value, coordinate):
    absolute_value = abs(float(value))
    degrees = int(np.floor(absolute_value))
    minutes = int(
        round(
            (absolute_value - degrees) * 60
        )
    )

    if minutes == 60:
        degrees += 1
        minutes = 0

    if coordinate == "longitude":
        direction = "E" if value >= 0 else "W"
    else:
        direction = "N" if value >= 0 else "S"

    return f"{degrees}°{minutes:02d}′{direction}"


def polygon_coordinates(geometry):
    x_coordinates = []
    y_coordinates = []

    if geometry.geom_type == "Polygon":
        polygons = [geometry]

    elif geometry.geom_type == "MultiPolygon":
        polygons = list(geometry.geoms)

    else:
        raise TypeError(
            f"Unsupported polygon geometry: {geometry.geom_type}"
        )

    for polygon in polygons:
        longitude, latitude = polygon.exterior.xy

        x_coordinates.extend(list(longitude))
        y_coordinates.extend(list(latitude))
        x_coordinates.append(None)
        y_coordinates.append(None)

    return x_coordinates, y_coordinates


def extract_line_coordinates(geodataframe):
    x_coordinates = []
    y_coordinates = []

    for geometry in geodataframe.geometry:
        if geometry is None or geometry.is_empty:
            continue

        if geometry.geom_type == "LineString":
            lines = [geometry]

        elif geometry.geom_type == "MultiLineString":
            lines = list(geometry.geoms)

        elif geometry.geom_type == "Polygon":
            lines = [geometry.exterior]

        elif geometry.geom_type == "MultiPolygon":
            lines = [
                polygon.exterior
                for polygon in geometry.geoms
            ]

        else:
            continue

        for line in lines:
            longitude, latitude = line.xy

            x_coordinates.extend(list(longitude))
            y_coordinates.extend(list(latitude))
            x_coordinates.append(None)
            y_coordinates.append(None)

    return x_coordinates, y_coordinates


def hex_to_rgba(hex_color, alpha):
    hex_color = hex_color.lstrip("#")

    red = int(hex_color[0:2], 16)
    green = int(hex_color[2:4], 16)
    blue = int(hex_color[4:6], 16)

    return (
        f"rgba({red},{green},{blue},{alpha})"
    )

In [ ]:
# ============================================================
# FIGURE 1. WHERE ARE THE 12 STUDY AREAS?
# ============================================================

regional_display = (
    municipalities
    .to_crs("EPSG:4326")
    .copy()
)

focused_display = (
    focused_municipalities
    .to_crs("EPSG:4326")
    .copy()
)

roads_display = (
    roads
    .to_crs("EPSG:4326")
    .copy()
)

haiyan_track_display = (
    haiyan_track
    .to_crs("EPSG:4326")
    .copy()
)


# ------------------------------------------------------------
# Display names and colours
# ------------------------------------------------------------

FOCUSED_DISPLAY_NAMES = {
    "Tacloban": "Tacloban City",
    "Ormoc": "Ormoc City",
    "Baybay": "Baybay City",
    "Catbalogan": "Catbalogan City",
    "Borongan": "Borongan City",
    "Guiuan": "Guiuan",
    "Palo": "Palo",
    "Tanauan": "Tanauan",
    "Tolosa": "Tolosa",
    "Dulag": "Dulag",
    "Basey": "Basey",
    "Lawaan": "Lawaan",
}

FOCUSED_COLORS = {
    "Tacloban": "#0057FF",
    "Ormoc": "#E76F51",
    "Baybay": "#2A9D8F",
    "Catbalogan": "#7B2CBF",
    "Borongan": "#F4A261",
    "Guiuan": "#D62828",
    "Palo": "#00A6A6",
    "Tanauan": "#3A86FF",
    "Tolosa": "#8338EC",
    "Dulag": "#FF006E",
    "Basey": "#6A994E",
    "Lawaan": "#BC6C25",
}


# ------------------------------------------------------------
# Common 16:9 map extent
# ------------------------------------------------------------

minimum_x, minimum_y, maximum_x, maximum_y = (
    focused_display.total_bounds
)

x_span = maximum_x - minimum_x
y_span = maximum_y - minimum_y

minimum_x -= x_span * 0.08
maximum_x += x_span * 0.08
minimum_y -= y_span * 0.08
maximum_y += y_span * 0.08

map_centre_x = (minimum_x + maximum_x) / 2
map_centre_y = (minimum_y + maximum_y) / 2

map_width = maximum_x - minimum_x
map_height = maximum_y - minimum_y

target_ratio = 16 / 9

if map_width / map_height < target_ratio:
    map_width = map_height * target_ratio
    minimum_x = map_centre_x - map_width / 2
    maximum_x = map_centre_x + map_width / 2
else:
    map_height = map_width / target_ratio
    minimum_y = map_centre_y - map_height / 2
    maximum_y = map_centre_y + map_height / 2

overview_bounds = (
    minimum_x,
    minimum_y,
    maximum_x,
    maximum_y,
)


# ------------------------------------------------------------
# Coordinate labels
# ------------------------------------------------------------

x_tick_values = np.arange(
    np.ceil(overview_bounds[0] * 2) / 2,
    np.floor(overview_bounds[2] * 2) / 2 + 0.001,
    0.5,
)

y_tick_values = np.arange(
    np.ceil(overview_bounds[1] * 2) / 2,
    np.floor(overview_bounds[3] * 2) / 2 + 0.001,
    0.5,
)

x_tick_labels = [
    degree_minute_label(value, "longitude")
    for value in x_tick_values
]

y_tick_labels = [
    degree_minute_label(value, "latitude")
    for value in y_tick_values
]


# ------------------------------------------------------------
# Regional municipality contours
# ------------------------------------------------------------

regional_overview = regional_display.cx[
    overview_bounds[0]:overview_bounds[2],
    overview_bounds[1]:overview_bounds[3],
].copy()

regional_boundary_x = []
regional_boundary_y = []

for geometry in regional_overview.geometry.boundary:
    if geometry.is_empty:
        continue

    lines = (
        list(geometry.geoms)
        if geometry.geom_type == "MultiLineString"
        else [geometry]
    )

    for line in lines:
        longitude, latitude = line.xy

        regional_boundary_x.extend(list(longitude))
        regional_boundary_y.extend(list(latitude))
        regional_boundary_x.append(None)
        regional_boundary_y.append(None)


# ------------------------------------------------------------
# Roads within the displayed extent
# ------------------------------------------------------------

roads_overview = roads_display.cx[
    overview_bounds[0]:overview_bounds[2],
    overview_bounds[1]:overview_bounds[3],
].copy()

if len(roads_overview) > 9000:
    roads_overview = roads_overview.iloc[
        ::max(1, len(roads_overview) // 9000)
    ].copy()

overview_road_x, overview_road_y = (
    extract_line_coordinates(roads_overview)
)


# ------------------------------------------------------------
# Haiyan path
# ------------------------------------------------------------

haiyan_track_x, haiyan_track_y = (
    extract_line_coordinates(haiyan_track_display)
)


# ------------------------------------------------------------
# Build figure
# ------------------------------------------------------------

fig = go.Figure()


# Background municipality contours
fig.add_trace(
    go.Scatter(
        x=regional_boundary_x,
        y=regional_boundary_y,
        mode="lines",
        line=dict(
            color="#D5DBE1",
            width=0.7,
        ),
        name="Municipality boundaries",
        legendgroup="map-context",
        hoverinfo="skip",
    )
)


# Roads
fig.add_trace(
    go.Scatter(
        x=overview_road_x,
        y=overview_road_y,
        mode="lines",
        line=dict(
            color="#C7D0DA",
            width=1.15,
        ),
        name="Roads",
        legendgroup="map-context",
        hoverinfo="skip",
    )
)


# Haiyan path
fig.add_trace(
    go.Scatter(
        x=haiyan_track_x,
        y=haiyan_track_y,
        mode="lines",
        line=dict(
            color="#111111",
            width=3.5,
        ),
        name="Haiyan Path",
        legendgroup="map-context",
        hovertemplate=(
            "Haiyan Path"
            "<extra></extra>"
        ),
    )
)


# Focused municipality polygons
for _, row in focused_display.iterrows():
    municipality_name = row["unit_name"]

    color = FOCUSED_COLORS[municipality_name]

    display_name = FOCUSED_DISPLAY_NAMES.get(
        municipality_name,
        municipality_name,
    )

    polygon_x, polygon_y = polygon_coordinates(
        row.geometry
    )

    fig.add_trace(
        go.Scatter(
            x=polygon_x,
            y=polygon_y,
            mode="lines",
            fill="toself",
            fillcolor=hex_to_rgba(color, 0.15),
            line=dict(
                color=color,
                width=2.4,
            ),
            name=display_name,
            legendgroup=(
                f"municipality-{municipality_name}"
            ),
            hovertemplate=(
                f"<b>{display_name}</b><br>"
                f"{row['province_name']}"
                "<extra></extra>"
            ),
        )
    )


# Municipality labels
for _, row in focused_display.iterrows():
    municipality_name = row["unit_name"]

    display_name = FOCUSED_DISPLAY_NAMES.get(
        municipality_name,
        municipality_name,
    )

    color = FOCUSED_COLORS[municipality_name]

    point = row.geometry.representative_point()

    fig.add_annotation(
        x=point.x,
        y=point.y,
        xref="x",
        yref="y",
        text=f"<b>{display_name}</b>",
        showarrow=False,
        font=dict(
            family="Arial",
            size=13,
            color=color,
        ),
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor=hex_to_rgba(color, 0.55),
        borderwidth=1,
        borderpad=3,
    )


# ------------------------------------------------------------
# Figure layout
# ------------------------------------------------------------

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1280,
    height=720,

    title=dict(
        text=(
            "<b>Focused Haiyan recovery study areas</b>"
            "<br>"
            "<sup>Original six locations and municipalities "
            "examined in the DEval case study</sup>"
        ),
        x=0.02,
        y=0.97,
        xanchor="left",
        yanchor="top",
        font=dict(
            size=24,
            color="#243B5A",
        ),
    ),

    xaxis=dict(
        title=None,
        range=[
            overview_bounds[0],
            overview_bounds[2],
        ],
        tickmode="array",
        tickvals=x_tick_values,
        ticktext=x_tick_labels,
        ticks="outside",
        ticklen=5,
        showgrid=True,
        gridcolor="rgba(36,59,90,0.11)",
        gridwidth=1,
        zeroline=False,
        constrain="domain",
    ),

    yaxis=dict(
        title=None,
        range=[
            overview_bounds[1],
            overview_bounds[3],
        ],
        tickmode="array",
        tickvals=y_tick_values,
        ticktext=y_tick_labels,
        ticks="outside",
        ticklen=5,
        showgrid=True,
        gridcolor="rgba(36,59,90,0.11)",
        gridwidth=1,
        zeroline=False,
        scaleanchor="x",
        scaleratio=1,
        constrain="domain",
    ),

    legend=dict(
        title=dict(
            text="<b>City / Municipality</b>",
            side="top",
            font=dict(size=16),
        ),
        orientation="h",
        x=0.5,
        y=-0.11,
        xanchor="center",
        yanchor="top",
        entrywidth=105,
        entrywidthmode="pixels",
        itemwidth=30,
        traceorder="normal",
        itemsizing="constant",
        font=dict(size=13),
        bgcolor="rgba(255,255,255,0.94)",
        bordercolor="rgba(36,59,90,0.20)",
        borderwidth=1,
    ),

    font=dict(
        family="Arial",
        size=14,
        color="#243B5A",
    ),

    margin=dict(
        l=90,
        r=30,
        t=100,
        b=130,
    ),
)

fig.show()

In [ ]:
# ============================================================
# 4. ALIGN GHSL, CLIP THE FOCUSED BOUNDARIES, AND RASTERIZE
# ============================================================

ghsl = rxr.open_rasterio(GHSL_PATH, masked=True)
if "band" in ghsl.dims:
    ghsl = ghsl.isel(band=0, drop=True)
if ghsl.rio.crs is None:
    raise ValueError("GHSL raster has no CRS.")

viirs_template = dnb.isel(date=0, drop=True)
ghsl_viirs = ghsl.rio.reproject_match(
    viirs_template, resampling=Resampling.nearest
).assign_coords(x=viirs_template.x, y=viirs_template.y)
ghsl_mask = ghsl_viirs.isin(GHSL_MASKS[SETTLEMENT_MASK]).fillna(False)

municipalities_raster_crs = focused_municipalities.to_crs(a2.rio.crs).copy()
municipalities_raster_crs["profile_id"] = np.arange(1, len(municipalities_raster_crs) + 1)

def rasterize_units(gdf, value_column):
    shapes = [(geometry, int(value)) for geometry, value in zip(gdf.geometry, gdf[value_column])]
    values = rasterize(
        shapes,
        out_shape=(dnb.sizes["y"], dnb.sizes["x"]),
        transform=dnb.rio.transform(recalc=True), fill=0,
        all_touched=False, dtype="int32",
    )
    return xr.DataArray(values, dims=SPATIAL_DIMS, coords={"y": dnb.y, "x": dnb.x})

all_zone_id = rasterize_units(municipalities_raster_crs, "profile_id")
study_mask = all_zone_id > 0
rq_base_mask = (ghsl_mask & study_mask).compute()
municipalities_display = municipalities_raster_crs.to_crs("EPSG:4326").copy()

zone_counts = []
for row in municipalities_raster_crs.itertuples():
    zone_counts.append({
        "profile_id": row.profile_id,
        "unit_name": row.unit_name,
        "g3_pixels": int(((all_zone_id == row.profile_id) & ghsl_mask).sum().item()),
    })
zone_counts = pd.DataFrame(zone_counts).sort_values("g3_pixels", ascending=False)

fig = px.bar(
    zone_counts, x="unit_name", y="g3_pixels", text="g3_pixels",
    labels={"unit_name": "Study area", "g3_pixels": "GHSL G3 pixels"},
    title="Reliability support before temporal filtering",
)
fig.update_traces(marker_color="#174A7E", textposition="outside")
fig.update_layout(
    template="plotly_white", paper_bgcolor="rgba(0,0,0,0)",
    width=1280, height=720, font=dict(family="Arial", size=14, color="#243B5A"),
    margin=dict(l=80, r=40, t=90, b=100), showlegend=False,
)
fig.show()

In [ ]:
# Copied from Notebook 3, source cell index 8
# ============================================================
# 7. BUILD THE RELIABILITY-QUALIFIED CUBE
# ============================================================

rq_unclipped = dnb.where((mqf == 0) & rq_base_mask)
rq_quantile_source = rq_unclipped

if hasattr(rq_unclipped.data, "rechunk"):
    spatial_axes = {
        rq_unclipped.get_axis_num(dimension): -1
        for dimension in SPATIAL_DIMS
    }
    rq_quantile_source = rq_unclipped.copy(
        data=rq_unclipped.data.rechunk(spatial_axes)
    )

rq_daily_p95 = (
    rq_quantile_source
    .quantile(RQ_CLIP_PERCENTILE / 100.0, dim=SPATIAL_DIMS, skipna=True)
    .squeeze(drop=True)
    .compute()
)

rq_cube = xr.where(
    rq_unclipped > rq_daily_p95,
    rq_daily_p95,
    rq_unclipped,
)
rq_cube.name = "reliability_qualified_ntl"

print(
    "Daily P95 range:",
    f"{float(rq_daily_p95.min(skipna=True)):.2f}",
    "to",
    f"{float(rq_daily_p95.max(skipna=True)):.2f}",
    "nW cm⁻² sr⁻¹",
)

In [ ]:
# ============================================================
# FIGURE 2. WHAT DOES DAILY P95 CAPPING CHANGE?
# ============================================================

quantile_audit = pd.DataFrame({
    "date": pd.DatetimeIndex(rq_daily_p95.date.values),
    "daily_p95": rq_daily_p95.values,
})

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=quantile_audit.date, y=quantile_audit.daily_p95,
    mode="lines", line=dict(color="#D55E00", width=1.5),
    name="Daily spatial P95",
    hovertemplate="%{x|%d %b %Y}<br>P95: %{y:.2f} nW cm⁻² sr⁻¹<extra></extra>",
))
fig.add_vline(x=EVENT_DATE, line_dash="dash", line_color="#0057FF", line_width=2)
fig.add_annotation(
    x=EVENT_DATE, y=1, yref="paper", text="Haiyan · 8 Nov 2013",
    showarrow=False, xanchor="left", yanchor="top",
    font=dict(color="#0057FF", size=12), bgcolor="rgba(255,255,255,0.8)",
)
fig.update_layout(
    template="plotly_white", paper_bgcolor="rgba(0,0,0,0)",
    width=1280, height=720,
    title=dict(text="Daily quantile cap applied to reliability-qualified radiance", x=0.02, xanchor="left"),
    xaxis_title="Date", yaxis_title="Daily spatial P95 (nW cm⁻² sr⁻¹)",
    font=dict(family="Arial", size=14, color="#243B5A"),
    margin=dict(l=90, r=40, t=95, b=75),
)
fig.show()

print(
    "Interpretation: values above each day's line are capped at that threshold. "
    "This limits leverage from extreme bright pixels while retaining the observation."
)

In [ ]:
# Copied from Notebook 3, source cell index 9
# ============================================================
# 8. PROFILE FUNCTIONS
# ============================================================


def crop_to_support(cube, support_mask):
    support_values = np.asarray(support_mask.fillna(False).values, dtype=bool)
    rows, columns = np.where(support_values)

    if len(rows) == 0:
        return None, None

    y_slice = slice(rows.min(), rows.max() + 1)
    x_slice = slice(columns.min(), columns.max() + 1)

    return (
        cube.isel(y=y_slice, x=x_slice),
        support_mask.isel(y=y_slice, x=x_slice),
    )


def build_pixel_matched_profile(
    cube,
    support_mask,
    aggregation_days,
    unit_name,
    unit_type,
    method="Reliability-qualified DNB-BRDF",
):
    '''Build a daily or non-overlapping multi-day profile.'''

    selected = (
        cube
        .sel(date=slice(ANALYSIS_START, PROFILE_END))
        .where(support_mask)
    )

    dates = pd.DatetimeIndex(selected["date"].values).normalize()
    block_numbers = np.floor_divide(
        (dates - EVENT_DATE).days,
        aggregation_days,
    ).astype(int)

    composites = (
        selected
        .assign_coords(block=("date", block_numbers))
        .groupby("block")
        .median(dim="date", skipna=True)
    )

    blocks = composites["block"].values.astype(int)
    block_start = EVENT_DATE + pd.to_timedelta(blocks * aggregation_days, unit="D")
    block_end = block_start + pd.Timedelta(days=aggregation_days - 1)

    baseline_blocks = blocks[
        (block_start >= BASELINE_START)
        & (block_end <= PRE_EVENT_END)
    ]

    if len(baseline_blocks) == 0:
        raise ValueError(f"{unit_name}: no complete baseline blocks were found.")

    expected_baseline_blocks = BASELINE_DAYS // aggregation_days
    if BASELINE_DAYS % aggregation_days != 0:
        raise ValueError(
            "BASELINE_DAYS must be divisible by aggregation_days so the "
            "baseline contains complete, non-overlapping composites."
        )
    if len(baseline_blocks) != expected_baseline_blocks:
        raise ValueError(
            f"{unit_name}: expected {expected_baseline_blocks} complete baseline "
            f"blocks inside {BASELINE_START.date()}–{PRE_EVENT_END.date()}, "
            f"but found {len(baseline_blocks)}."
        )

    baseline_composites = composites.sel(block=baseline_blocks).compute()
    baseline_observations = baseline_composites.notnull().sum(dim="block")

    ntl0 = baseline_composites.median(dim="block", skipna=True)
    baseline_quantiles = baseline_composites.quantile(
        [0.25, 0.75],
        dim="block",
        skipna=True,
    )
    ntl0_q25 = baseline_quantiles.sel(quantile=0.25, drop=True)
    ntl0_q75 = baseline_quantiles.sel(quantile=0.75, drop=True)

    minimum_baseline = MIN_BASELINE_OBSERVATIONS[aggregation_days]
    fixed_mask = (
        support_mask
        & (baseline_observations >= minimum_baseline)
        & np.isfinite(ntl0)
        & (ntl0 > 0)
    ).compute()

    fixed_pixel_count = int(fixed_mask.sum().item())

    if fixed_pixel_count == 0:
        raise ValueError(
            f"{unit_name}: no baseline-lit {SETTLEMENT_MASK} pixels "
            f"met the baseline requirement."
        )

    paired_valid = composites.notnull() & fixed_mask & ntl0.notnull()
    valid_pixel_count = paired_valid.sum(dim=SPATIAL_DIMS)
    spatial_coverage_pct = 100.0 * valid_pixel_count / fixed_pixel_count

    current_radiance = composites.where(paired_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    matched_baseline = ntl0.where(paired_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    recovery_pct = 100.0 * current_radiance / matched_baseline

    # Raw magnitude before baseline normalization: spatial median across the
    # currently valid, fixed baseline-lit pixels.
    raw_median_source = composites.where(paired_valid)
    matched_median_source = ntl0.where(paired_valid)

    if hasattr(raw_median_source.data, "rechunk"):
        raw_spatial_axes = {
            raw_median_source.get_axis_num(dimension): -1
            for dimension in SPATIAL_DIMS
        }
        raw_median_source = raw_median_source.copy(
            data=raw_median_source.data.rechunk(raw_spatial_axes)
        )
    if hasattr(matched_median_source.data, "rechunk"):
        matched_spatial_axes = {
            matched_median_source.get_axis_num(dimension): -1
            for dimension in SPATIAL_DIMS
        }
        matched_median_source = matched_median_source.copy(
            data=matched_median_source.data.rechunk(matched_spatial_axes)
        )

    raw_median_ntl = raw_median_source.median(
        dim=SPATIAL_DIMS,
        skipna=True,
    )
    matched_baseline_median_ntl = matched_median_source.median(
        dim=SPATIAL_DIMS,
        skipna=True,
    )

    uncertainty_valid = (
        paired_valid
        & ntl0_q25.notnull()
        & ntl0_q75.notnull()
        & (ntl0_q25 > 0)
    )
    uncertainty_current = composites.where(uncertainty_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    baseline_q25_sum = ntl0_q25.where(uncertainty_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    baseline_q75_sum = ntl0_q75.where(uncertainty_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )

    recovery_low_pct = 100.0 * uncertainty_current / baseline_q75_sum
    recovery_high_pct = 100.0 * uncertainty_current / baseline_q25_sum

    reduced = xr.Dataset(
        {
            "current_radiance": current_radiance,
            "matched_baseline_radiance": matched_baseline,
            "raw_median_ntl": raw_median_ntl,
            "matched_baseline_median_ntl": matched_baseline_median_ntl,
            "recovery_pct": recovery_pct,
            "recovery_low_pct": recovery_low_pct,
            "recovery_high_pct": recovery_high_pct,
            "spatial_coverage_pct": spatial_coverage_pct,
            "valid_pixel_count": valid_pixel_count,
        }
    ).compute()

    profile = (
        reduced
        .to_dataframe()
        .reset_index()
        .sort_values("block")
        .reset_index(drop=True)
    )
    profile["date_start"] = EVENT_DATE + pd.to_timedelta(
        profile["block"] * aggregation_days,
        unit="D",
    )
    profile["date_end"] = profile["date_start"] + pd.Timedelta(
        days=aggregation_days - 1
    )
    profile["date_mid"] = profile["date_start"] + pd.to_timedelta(
        (aggregation_days - 1) / 2,
        unit="D",
    )

    below_gate = profile["spatial_coverage_pct"] < SPATIAL_COMPLETENESS_PCT
    profile.loc[
        below_gate,
        [
            "current_radiance",
            "matched_baseline_radiance",
            "raw_median_ntl",
            "matched_baseline_median_ntl",
            "recovery_pct",
            "recovery_low_pct",
            "recovery_high_pct",
        ],
    ] = np.nan

    profile["observation_status"] = np.where(
        profile["recovery_pct"].notna(),
        "observed",
        "not observable",
    )
    profile["unit_name"] = unit_name
    profile["unit_type"] = unit_type
    profile["method"] = method
    profile["aggregation_days"] = aggregation_days
    profile["ghsl_mask"] = SETTLEMENT_MASK
    profile["sc_threshold_pct"] = SPATIAL_COMPLETENESS_PCT
    profile["baseline_start"] = BASELINE_START
    profile["baseline_end"] = PRE_EVENT_END
    profile["baseline_days"] = BASELINE_DAYS
    profile["temporal_composite_statistic"] = "median"
    profile["raw_spatial_statistic"] = "median"
    profile["baseline_definition"] = (
        "per-pixel median of complete median reliability-qualified composites "
        "within the 60 days before Haiyan"
    )

    baseline_rows = profile.loc[
        (profile["date_start"] >= BASELINE_START)
        & (profile["date_end"] <= PRE_EVENT_END)
    ]

    report = {
        "unit_name": unit_name,
        "unit_type": unit_type,
        "aggregation_days": aggregation_days,
        "ghsl_mask": SETTLEMENT_MASK,
        "ghsl_classes": str(GHSL_MASKS[SETTLEMENT_MASK]),
        "sc_threshold_pct": SPATIAL_COMPLETENESS_PCT,
        "baseline_start": BASELINE_START.date(),
        "baseline_end": PRE_EVENT_END.date(),
        "baseline_blocks": len(baseline_blocks),
        "minimum_baseline_observations": minimum_baseline,
        "temporal_composite_statistic": "median",
        "raw_spatial_statistic": "median",
        "fixed_baseline_pixels": fixed_pixel_count,
        "median_baseline_coverage_pct": baseline_rows[
            "spatial_coverage_pct"
        ].median(),
    }

    return profile, composites.where(fixed_mask), ntl0.where(fixed_mask), fixed_mask, report

In [ ]:
# ============================================================
# 5. MUNICIPALITY-WIDE PROFILES, BASELINES, AND COMPOSITES
# ============================================================

municipality_profiles = []
municipality_reports = []
municipality_failures = []
baseline_surfaces = {}
fixed_masks = {}
municipality_composites = {}

for row in municipalities_raster_crs.itertuples():
    support = (all_zone_id == int(row.profile_id)) & ghsl_mask
    local_cube, local_support = crop_to_support(rq_cube, support)
    if local_cube is None:
        municipality_failures.append({"unit_name": row.unit_name, "reason": "No G3 pixels"})
        continue
    try:
        profile, composites, baseline_surface, fixed_mask, report = build_pixel_matched_profile(
            local_cube, local_support, AGGREGATION_DAYS, row.unit_name, "Municipality-wide G3"
        )
        profile["profile_id"] = int(row.profile_id)
        municipality_profiles.append(profile)
        municipality_composites[row.unit_name] = composites.compute()
        baseline_surfaces[int(row.profile_id)] = baseline_surface.compute()
        fixed_masks[int(row.profile_id)] = fixed_mask.compute()
        fixed_values = baseline_surface.where(fixed_mask).values
        fixed_values = fixed_values[np.isfinite(fixed_values)]
        report.update({
            "profile_id": int(row.profile_id),
            "unit_name": row.unit_name,
            "baseline_mean_ntl": float(np.mean(fixed_values)),
            "baseline_median_ntl": float(np.median(fixed_values)),
            "baseline_total_ntl": float(np.sum(fixed_values)),
        })
        municipality_reports.append(report)
    except ValueError as error:
        municipality_failures.append({"unit_name": row.unit_name, "reason": str(error)})

municipality_four_day = pd.concat(municipality_profiles, ignore_index=True)
municipality_reports = pd.DataFrame(municipality_reports)
municipality_failures = pd.DataFrame(municipality_failures)

print(f"Municipality-wide profiles produced: {municipality_four_day.unit_name.nunique()} / {len(STUDY_AREAS)}")
if not municipality_failures.empty:
    display(municipality_failures)

In [ ]:
# ============================================================
# 6. SYSTEMATIC 5×5 PROFILES
# ============================================================

def kernel_slices_from_coordinate(x_value, y_value, size=5):
    if size % 2 != 1:
        raise ValueError("KERNEL_SIZE must be odd.")
    x_index = int(np.abs(dnb.x.values - x_value).argmin())
    y_index = int(np.abs(dnb.y.values - y_value).argmin())
    half = size // 2
    return (
        slice(max(0, y_index - half), min(dnb.sizes["y"], y_index + half + 1)),
        slice(max(0, x_index - half), min(dnb.sizes["x"], x_index + half + 1)),
    )

kernel_profiles = []
kernel_reports = []
kernel_anchors = []

for row in municipalities_raster_crs.itertuples():
    profile_id = int(row.profile_id)
    if profile_id not in baseline_surfaces:
        continue
    baseline_surface = baseline_surfaces[profile_id]
    values = np.asarray(baseline_surface.values, dtype=float)
    local_y, local_x = np.unravel_index(np.nanargmax(values), values.shape)
    anchor_x = float(baseline_surface.x.values[local_x])
    anchor_y = float(baseline_surface.y.values[local_y])
    y_slice, x_slice = kernel_slices_from_coordinate(anchor_x, anchor_y, KERNEL_SIZE)
    local_cube = rq_cube.isel(y=y_slice, x=x_slice)
    local_support = ((all_zone_id == profile_id) & ghsl_mask).isel(y=y_slice, x=x_slice)
    support_pixels = int(local_support.sum().item())
    kernel_anchors.append({
        "profile_id": profile_id, "unit_name": row.unit_name,
        "anchor_x": anchor_x, "anchor_y": anchor_y,
        "anchor_baseline_ntl": float(values[local_y, local_x]),
        "g3_municipal_pixels_in_window": support_pixels,
    })
    if support_pixels == 0:
        continue
    try:
        profile, _, baseline_5x5, fixed_5x5, report = build_pixel_matched_profile(
            local_cube, local_support, AGGREGATION_DAYS, row.unit_name, "Local 5×5 window"
        )
        profile["profile_id"] = profile_id
        kernel_profiles.append(profile)
        fixed_values = baseline_5x5.where(fixed_5x5).values
        fixed_values = fixed_values[np.isfinite(fixed_values)]
        report.update({
            "profile_id": profile_id, "unit_name": row.unit_name,
            "baseline_mean_ntl": float(np.mean(fixed_values)),
            "baseline_median_ntl": float(np.median(fixed_values)),
            "baseline_total_ntl": float(np.sum(fixed_values)),
            "anchor_x": anchor_x, "anchor_y": anchor_y,
            "anchor_baseline_ntl": float(values[local_y, local_x]),
        })
        kernel_reports.append(report)
    except ValueError:
        continue

kernel_four_day = pd.concat(kernel_profiles, ignore_index=True)
kernel_reports = pd.DataFrame(kernel_reports)
kernel_anchors = pd.DataFrame(kernel_anchors)

fig = px.bar(
    kernel_anchors.sort_values("anchor_baseline_ntl"),
    x="unit_name", y="anchor_baseline_ntl",
    color="g3_municipal_pixels_in_window", text="g3_municipal_pixels_in_window",
    color_continuous_scale="Blues",
    labels={
        "unit_name": "Study area", "anchor_baseline_ntl": "Anchor baseline radiance",
        "g3_municipal_pixels_in_window": "G3 pixels in 5×5",
    },
    title="What each 5×5 kernel is centred on",
)
fig.update_layout(
    template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", width=1280, height=720,
    font=dict(family="Arial", size=14, color="#243B5A"),
    margin=dict(l=80, r=40, t=90, b=100),
)
fig.show()

In [ ]:
# Copied from Notebook 3, source cell index 13
# ============================================================
# 12. RECOVERY METRIC FUNCTIONS
# ============================================================


def expected_event_blocks(aggregation_days=4):
    inclusive_days = (PROFILE_END - EVENT_DATE).days + 1
    return int(np.ceil(inclusive_days / aggregation_days))


def longest_missing_run_days(profile, aggregation_days=4):
    expected_blocks = np.arange(
        0,
        int(np.floor((PROFILE_END - EVENT_DATE).days / aggregation_days)) + 1,
    )
    observed_blocks = set(
        profile.loc[
            (profile["date_start"] >= EVENT_DATE)
            & (profile["date_start"] <= PROFILE_END)
            & profile["recovery_pct"].notna(),
            "block",
        ].astype(int)
    )
    missing = np.array(
        [block not in observed_blocks for block in expected_blocks],
        dtype=int,
    )

    longest = current = 0
    for value in missing:
        current = current + 1 if value else 0
        longest = max(longest, current)

    return longest * aggregation_days


def observability_class(profile, aggregation_days=4):
    post = profile.loc[
        (profile["date_start"] >= EVENT_DATE)
        & (profile["date_start"] <= PROFILE_END)
    ]
    retained = int(post["recovery_pct"].notna().sum())
    expected = expected_event_blocks(aggregation_days)
    retained_pct = 100.0 * retained / expected if expected else np.nan
    max_gap = longest_missing_run_days(profile, aggregation_days)

    if (
        retained >= MIN_EVENT_COMPOSITES
        and retained_pct >= MIN_EVENT_RETENTION_PCT
        and max_gap <= MAX_INTERPRETABLE_GAP_DAYS
    ):
        label, flag = "interpretable with interval timing", "OBS_OK"
    elif retained >= MIN_EVENT_COMPOSITES and retained_pct >= MIN_EVENT_RETENTION_PCT:
        label, flag = "observation-limited", "OBS_LIMITED"
    else:
        label, flag = "not observable", "NOT_OBSERVABLE"

    return {
        "retained_composites": retained,
        "expected_composites": expected,
        "retained_pct": retained_pct,
        "max_gap_days": max_gap,
        "observability": label,
        "quality_flag": flag,
    }


def persistent_crossing(
    profile,
    threshold,
    value_column="recovery_pct",
    start_block=0,
):
    """First persistent threshold return at or after the observed impact block."""

    post = (
        profile.loc[
            (profile["date_start"] >= EVENT_DATE)
            & (profile["date_start"] <= PROFILE_END)
            & (profile["block"] >= start_block)
            & profile[value_column].notna()
        ]
        .sort_values("block")
        .set_index("block", drop=False)
    )

    for block, row in post.iterrows():
        required = list(range(int(block), int(block) + PERSISTENCE_BLOCKS))
        if not set(required).issubset(post.index):
            continue
        if not (post.loc[required, value_column] >= threshold).all():
            continue

        previous_below = post.loc[
            (post["block"] < block) & (post[value_column] < threshold)
        ]
        lower_day = int((row["date_start"] - EVENT_DATE).days)
        if not previous_below.empty:
            lower_day = max(
                0,
                int(
                    (
                        previous_below.iloc[-1]["date_end"]
                        + pd.Timedelta(days=1)
                        - EVENT_DATE
                    ).days
                ),
            )

        upper_day = int((row["date_start"] - EVENT_DATE).days)
        return {
            "day": upper_day,
            "lower_day": lower_day,
            "upper_day": upper_day,
            "date": row["date_start"],
            "observed_value": float(row[value_column]),
        }

    return None


def format_interval(crossing):
    if crossing is None:
        return None
    return f"{crossing['lower_day']}–{crossing['upper_day']} d"


def format_baseline_range(optimistic, conservative):
    if optimistic is None and conservative is None:
        return None
    earliest = optimistic["day"] if optimistic is not None else None
    latest = conservative["day"] if conservative is not None else None
    if earliest is not None and latest is not None:
        return f"{min(earliest, latest)}–{max(earliest, latest)} d"
    if earliest is not None:
        return f"≥{earliest} d; conservative crossing absent"
    return f"≤{latest} d; optimistic crossing absent"


def calculate_recovery_metrics(profile):
    """Calculate impact and persistent return-to-baseline milestones.

    T50/T80/T90 mean the first of two consecutive admissible four-day
    composites at or above 50/80/90% of the matched 60-day baseline,
    searched only after the observed Stage-1 nadir. They are not fractions
    of the shock-to-baseline amplitude.
    """

    profile = profile.sort_values("date_start").copy()
    obs = observability_class(profile, aggregation_days=4)
    event = profile.loc[
        (profile["date_start"] >= EVENT_DATE)
        & (profile["date_start"] <= PROFILE_END)
    ]
    impact_window = event.loc[
        event["date_start"] <= EVENT_DATE + pd.Timedelta(days=59)
    ].dropna(subset=["recovery_pct"])

    impact_row = None
    if impact_window.empty:
        impact_recovery = impact_drop = np.nan
        impact_drop_low = impact_drop_high = np.nan
        impact_date, impact_block = pd.NaT, np.nan
    else:
        impact_row = impact_window.loc[impact_window["recovery_pct"].idxmin()]
        impact_recovery = float(impact_row["recovery_pct"])
        impact_date = impact_row["date_start"]
        impact_block = int(impact_row["block"])
        impact_drop = 100.0 - impact_recovery
        impact_drop_low = 100.0 - float(impact_row["recovery_high_pct"])
        impact_drop_high = 100.0 - float(impact_row["recovery_low_pct"])

    crossings = {}
    for threshold in (50, 80, 90):
        threshold_lost = bool(
            impact_row is not None and impact_recovery < threshold
        )

        if threshold_lost:
            central = persistent_crossing(
                profile,
                threshold,
                "recovery_pct",
                start_block=impact_block,
            )
            optimistic_lost = (
                pd.notna(impact_row["recovery_high_pct"])
                and float(impact_row["recovery_high_pct"]) < threshold
            )
            conservative_lost = (
                pd.notna(impact_row["recovery_low_pct"])
                and float(impact_row["recovery_low_pct"]) < threshold
            )
            optimistic = (
                persistent_crossing(
                    profile,
                    threshold,
                    "recovery_high_pct",
                    start_block=impact_block,
                )
                if optimistic_lost
                else None
            )
            conservative = (
                persistent_crossing(
                    profile,
                    threshold,
                    "recovery_low_pct",
                    start_block=impact_block,
                )
                if conservative_lost
                else None
            )
        else:
            central = optimistic = conservative = None

        if impact_row is None:
            status = "not observable in the 0–59 day impact window"
        elif not threshold_lost:
            status = "threshold not lost at observed nadir"
        elif central is not None:
            status = "supported; persistent return after observed nadir"
        elif obs["quality_flag"] == "NOT_OBSERVABLE":
            status = "not observable"
        elif obs["quality_flag"] == "OBS_LIMITED":
            status = "not identifiable: observation-limited"
        else:
            status = f"not recovered by {(PROFILE_END - EVENT_DATE).days} d"

        crossings[threshold] = {
            "central": central,
            "optimistic": optimistic,
            "conservative": conservative,
            "lost": threshold_lost,
            "status": status,
        }

    post_observed = event.dropna(subset=["recovery_pct"]).copy()
    slope = np.nan
    if not post_observed.empty and pd.notna(impact_date):
        slope_end_day = (
            crossings[90]["central"]["day"]
            if crossings[90]["central"] is not None
            else int((PROFILE_END - EVENT_DATE).days)
        )
        slope_data = post_observed.loc[
            (post_observed["date_start"] >= impact_date)
            & (
                post_observed["date_start"]
                <= EVENT_DATE + pd.Timedelta(days=slope_end_day)
            )
        ]
        if len(slope_data) >= 3:
            x = (slope_data["date_start"] - EVENT_DATE).dt.days.to_numpy(dtype=float)
            y = slope_data["recovery_pct"].to_numpy(dtype=float)
            slope = float(np.polyfit(x, y, 1)[0])

    stage3 = event.loc[
        (event["date_start"] >= EVENT_DATE + pd.Timedelta(days=120))
        & (event["date_start"] <= EVENT_DATE + pd.Timedelta(days=179))
    ].dropna(subset=["recovery_pct"])
    stability_mad = (
        float(np.median(np.abs(stage3["recovery_pct"] - stage3["recovery_pct"].median())))
        if not stage3.empty
        else np.nan
    )
    stable_share = (
        float(stage3["recovery_pct"].between(90, 110).sum() * 100.0 / len(stage3))
        if not stage3.empty
        else np.nan
    )

    result = {
        "unit_name": profile["unit_name"].iloc[0],
        "unit_type": profile["unit_type"].iloc[0],
        "metric_definition": (
            "persistent return to 50/80/90% of the matched 60-day baseline "
            "after the observed 0–59 day nadir"
        ),
        "ghsl_mask": SETTLEMENT_MASK,
        "ghsl_classes": str(GHSL_MASKS[SETTLEMENT_MASK]),
        "sc_threshold_pct": SPATIAL_COMPLETENESS_PCT,
        "baseline_days": BASELINE_DAYS,
        "baseline_start": BASELINE_START.date(),
        "baseline_end": PRE_EVENT_END.date(),
        "impact_date": impact_date,
        "impact_block": impact_block,
        "impact_recovery_pct": impact_recovery,
        "impact_drop_pct": impact_drop,
        "impact_drop_range_pct": (
            f"{impact_drop_low:.1f}–{impact_drop_high:.1f}"
            if np.isfinite(impact_drop_low) and np.isfinite(impact_drop_high)
            else None
        ),
        "recovery_slope_pct_per_day": slope,
        "observed_days_below_baseline": int(
            (post_observed["recovery_pct"] < 100).sum() * 4
        ),
        "stage3_stability_mad_pct": stability_mad,
        "stage3_within_90_110_pct": stable_share,
        "median_event_sc_pct": float(event["spatial_coverage_pct"].median()),
        "minimum_event_sc_pct": float(event["spatial_coverage_pct"].min()),
        **obs,
    }

    for threshold, crossing in crossings.items():
        result[f"T{threshold}_threshold_lost"] = crossing["lost"]
        result[f"T{threshold}_day"] = (
            crossing["central"]["day"]
            if crossing["central"] is not None
            else np.nan
        )
        result[f"T{threshold}_observation_interval"] = format_interval(
            crossing["central"]
        )
        result[f"T{threshold}_baseline_range"] = format_baseline_range(
            crossing["optimistic"],
            crossing["conservative"],
        )
        result[f"T{threshold}_status"] = crossing["status"]

    return result

In [ ]:
# ============================================================
# 7. RECOVERY METRICS FOR BOTH SUPPORTS
# ============================================================

def metric_table(profiles):
    return pd.DataFrame([
        calculate_recovery_metrics(group)
        for _, group in profiles.groupby("unit_name", sort=True, observed=True)
    ])

municipality_metrics = metric_table(municipality_four_day)
kernel_metrics = metric_table(kernel_four_day)

In [ ]:
# ============================================================
# 8. NOTEBOOK 3 VISUAL LANGUAGE
# ============================================================

PLOT_FONT = "Arial"
PLOT_TEXT_COLOR = "#243B5A"
PLOT_GRID_COLOR = "#E8EDF3"
EVENT_COLOR = "#0057FF"
BASELINE_COLOR = "#6C7882"
AREA_COLORS = dict(zip(STUDY_AREAS, px.colors.qualitative.Dark24[:len(STUDY_AREAS)]))
FAMILY_COLORS = ["#0072B2", "#E69F00", "#D55E00", "#009E73"]

def style_figure(fig, title, legend_y=1.10):
    fig.update_layout(
        template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="white",
        title=dict(text=title, x=0.02, xanchor="left", font=dict(size=24, color=PLOT_TEXT_COLOR)),
        font=dict(family=PLOT_FONT, size=14, color=PLOT_TEXT_COLOR),
        width=1280, height=720,
        margin=dict(l=85, r=45, t=115, b=75),
        legend=dict(orientation="h", y=legend_y, x=0, xanchor="left", yanchor="bottom"),
        hovermode="closest",
    )
    fig.update_xaxes(showgrid=True, gridcolor=PLOT_GRID_COLOR, zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor=PLOT_GRID_COLOR, zeroline=False)
    return fig

def add_haiyan_marker(fig, row=None, col=None):
    kwargs = {} if row is None else {"row": row, "col": col}
    fig.add_vline(x=EVENT_DATE, line_dash="dash", line_color=EVENT_COLOR, line_width=2, **kwargs)

def boundary_coordinates(geometry):
    boundary = geometry.boundary
    lines = boundary.geoms if boundary.geom_type == "MultiLineString" else [boundary]
    x_values, y_values = [], []
    for line in lines:
        x, y = line.xy
        x_values.extend([*x, None])
        y_values.extend([*y, None])
    return x_values, y_values

## Tacloban worked example: from pixels to one recovery metric

The next figures intentionally stay with one city. They show the exact transformation before the same logic is repeated for the other study areas.

1. Inspect the fixed spatial support and 5×5 anchor.
2. Compare the pixel-level baseline, early shock and later recovery patterns.
3. Aggregate only the currently observed fixed pixels into a baseline-relative trajectory.
4. identify the early post-event nadir and persistent T50/T80/T90 returns.
5. Compare municipality-wide and 5×5 results to determine whether the local core represents the broader city.

In [ ]:
# ============================================================
# FIGURE 3. TACLOBAN SUPPORT AND THREE SPATIAL STATES
# ============================================================

EXAMPLE_AREA = "Tacloban"
example_row = municipalities_raster_crs.loc[
    municipalities_raster_crs.unit_name.eq(EXAMPLE_AREA)
].iloc[0]
example_id = int(example_row.profile_id)
example_baseline = baseline_surfaces[example_id]
example_fixed = fixed_masks[example_id]
example_composites = municipality_composites[EXAMPLE_AREA]
example_profile = municipality_four_day.loc[
    municipality_four_day.unit_name.eq(EXAMPLE_AREA)
].copy()

observed_post = example_profile.loc[
    example_profile.date_start.ge(EVENT_DATE) & example_profile.recovery_pct.notna()
]
shock_block = int(observed_post.loc[observed_post.recovery_pct.idxmin(), "block"])
recovery_candidates = observed_post.loc[observed_post.block.ge(shock_block + 10)]
recovery_block = int((recovery_candidates.recovery_pct - 100).abs().idxmin())
recovery_block = int(example_profile.loc[recovery_block, "block"])

maps = [
    ("Fixed 60-day baseline", example_baseline.where(example_fixed)),
    (f"Early shock · block {shock_block}", example_composites.sel(block=shock_block)),
    (f"Later recovery · block {recovery_block}", example_composites.sel(block=recovery_block)),
]

map_values = np.concatenate([
    np.asarray(layer.values, dtype=float)[np.isfinite(layer.values)] for _, layer in maps
])
map_color_max = max(float(np.nanquantile(map_values, MAP_DISPLAY_QUANTILE)), 1.0)
boundary_x, boundary_y = boundary_coordinates(example_row.geometry)

fig = make_subplots(rows=1, cols=3, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[title for title, _ in maps], horizontal_spacing=0.035)
for column, (_, layer) in enumerate(maps, start=1):
    land = xr.where((all_zone_id == example_id), 1.0, np.nan)
    fig.add_trace(go.Heatmap(
        x=land.x, y=land.y, z=land.values,
        colorscale=[[0, "#D7D7D7"], [1, "#D7D7D7"]],
        showscale=False, hoverinfo="skip",
    ), row=1, col=column)
    fig.add_trace(go.Heatmap(
        x=layer.x, y=layer.y, z=layer.values,
        coloraxis="coloraxis", zsmooth=False, hoverongaps=False,
        hovertemplate="DNB-BRDF: %{z:.2f} nW cm⁻² sr⁻¹<extra></extra>",
    ), row=1, col=column)
    fig.add_trace(go.Scatter(
        x=boundary_x, y=boundary_y, mode="lines",
        line=dict(color="#202020", width=1.4),
        hoverinfo="skip", showlegend=False,
    ), row=1, col=column)
    fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=column)
    fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, scaleanchor=f"x{column if column > 1 else ''}", row=1, col=column)

fig.update_layout(
    template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", width=1280, height=720,
    title=dict(text=f"{EXAMPLE_AREA}: baseline → shock → recovery", x=0.02, xanchor="left"),
    coloraxis=dict(
        colorscale="Inferno", cmin=0, cmax=map_color_max,
        colorbar=dict(title="DNB-BRDF<br>nW cm⁻² sr⁻¹", thickness=18),
    ),
    font=dict(family="Arial", size=14, color="#243B5A"),
    margin=dict(l=40, r=105, t=100, b=40),
)
fig.show()

In [ ]:
# ============================================================
# FIGURE 4. TACLOBAN TEMPORAL PROFILE AND MILESTONES
# ============================================================

example_metric = municipality_metrics.loc[municipality_metrics.unit_name.eq(EXAMPLE_AREA)].iloc[0]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=example_profile.date_start, y=example_profile.recovery_pct,
    mode="lines+markers", connectgaps=False,
    line=dict(color=AREA_COLORS[EXAMPLE_AREA], width=2.5), marker=dict(size=5),
    customdata=np.column_stack([
        example_profile.spatial_coverage_pct,
        example_profile.valid_pixel_count,
    ]),
    name="Municipality-wide G3",
    hovertemplate=(
        "%{x|%d %b %Y}<br>Recovery: %{y:.1f}%"
        "<br>Spatial completeness: %{customdata[0]:.1f}%"
        "<br>Observed pixels: %{customdata[1]:.0f}<extra></extra>"
    ),
))
fig.add_hline(y=100, line_dash="dot", line_color=BASELINE_COLOR)
add_haiyan_marker(fig)

for threshold, color in [(50, "#56B4E9"), (80, "#009E73"), (90, "#D55E00")]:
    day = example_metric[f"T{threshold}_day"]
    if pd.notna(day):
        date = EVENT_DATE + pd.Timedelta(days=float(day))
        fig.add_trace(go.Scatter(
            x=[date], y=[threshold], mode="markers+text",
            marker=dict(size=11, color=color, line=dict(color="white", width=1)),
            text=[f"T{threshold}: {int(day)} d"], textposition="top center",
            name=f"T{threshold}", hovertemplate=f"T{threshold}: {int(day)} days<extra></extra>",
        ))

style_figure(fig, f"{EXAMPLE_AREA}: observed trajectory translated into persistent recovery milestones")
fig.update_yaxes(title="Recovery (% of matched baseline)")
fig.update_xaxes(title="Date")
fig.show()

In [ ]:
# ============================================================
# FIGURE 5. TACLOBAN: MUNICIPALITY-WIDE VERSUS 5×5
# ============================================================

comparison = pd.concat([
    municipality_four_day.loc[municipality_four_day.unit_name.eq(EXAMPLE_AREA)],
    kernel_four_day.loc[kernel_four_day.unit_name.eq(EXAMPLE_AREA)],
], ignore_index=True)

fig = px.line(
    comparison, x="date_start", y="recovery_pct", color="unit_type",
    markers=True, line_shape="linear",
    custom_data=["spatial_coverage_pct", "valid_pixel_count"],
    labels={"date_start": "Date", "recovery_pct": "Recovery (% baseline)", "unit_type": "Spatial support"},
)
fig.update_traces(connectgaps=False, hovertemplate=(
    "%{fullData.name}<br>%{x|%d %b %Y}<br>Recovery: %{y:.1f}%"
    "<br>Spatial completeness: %{customdata[0]:.1f}%"
    "<br>Observed pixels: %{customdata[1]:.0f}<extra></extra>"
))
fig.add_hline(y=100, line_dash="dot", line_color=BASELINE_COLOR)
add_haiyan_marker(fig)
style_figure(fig, f"{EXAMPLE_AREA}: does the 5×5 core tell the same story as the whole city?")
fig.show()

## Repeat the same translation for all 12 areas

The next views preserve the individual trajectories before summarising them. Missing periods remain visible. T50/T80/T90 are shown only where the threshold was genuinely lost and a persistent return was observed; an absent point is not automatically “no recovery.” Its status must be checked against observability.

In [ ]:
# ============================================================
# FIGURE 6. SMALL MULTIPLES FOR ALL AREAS AND BOTH SUPPORTS
# ============================================================

for support_name, profiles in [
    ("Municipality-wide G3", municipality_four_day),
    ("Local 5×5 window", kernel_four_day),
]:
    available = [name for name in STUDY_AREAS if name in set(profiles.unit_name)]
    fig = make_subplots(rows=3, cols=4, subplot_titles=available, shared_xaxes=True, shared_yaxes=True,
                        horizontal_spacing=0.045, vertical_spacing=0.10)
    for index, name in enumerate(available):
        row, column = divmod(index, 4)
        group = profiles.loc[profiles.unit_name.eq(name)]
        fig.add_trace(go.Scatter(
            x=group.date_start, y=group.recovery_pct,
            mode="lines", connectgaps=False,
            line=dict(color=AREA_COLORS[name], width=2),
            name=name, showlegend=False,
            customdata=np.column_stack([group.spatial_coverage_pct, group.valid_pixel_count]),
            hovertemplate=(
                f"{name}<br>%{{x|%d %b %Y}}<br>Recovery: %{{y:.1f}}%"
                "<br>SC: %{customdata[0]:.1f}%<br>Pixels: %{customdata[1]:.0f}<extra></extra>"
            ),
        ), row=row + 1, col=column + 1)
        fig.add_hline(y=100, line_dash="dot", line_color=BASELINE_COLOR, row=row + 1, col=column + 1)
        add_haiyan_marker(fig, row=row + 1, col=column + 1)
    style_figure(fig, f"Individual recovery trajectories · {support_name}")
    fig.update_layout(showlegend=False)
    fig.update_yaxes(range=[0, 180])
    fig.show()

In [ ]:
# ============================================================
# FIGURE 7. T50, T80, AND T90 WITHOUT A SUMMARY TABLE
# ============================================================

def milestone_long(metrics, support):
    rows = []
    for record in metrics.to_dict("records"):
        for threshold in (50, 80, 90):
            rows.append({
                "unit_name": record["unit_name"], "support": support,
                "threshold": f"T{threshold}", "day": record[f"T{threshold}_day"],
                "status": record[f"T{threshold}_status"],
                "quality_flag": record["quality_flag"],
            })
    return pd.DataFrame(rows)

milestones = pd.concat([
    milestone_long(municipality_metrics, "Municipality-wide G3"),
    milestone_long(kernel_metrics, "Local 5×5 window"),
], ignore_index=True)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Municipality-wide G3", "Local 5×5 window"),
                    shared_yaxes=True, horizontal_spacing=0.08)
threshold_colors = {"T50": "#56B4E9", "T80": "#009E73", "T90": "#D55E00"}
threshold_symbols = {"T50": "circle", "T80": "square", "T90": "diamond"}
for column, support in enumerate(["Municipality-wide G3", "Local 5×5 window"], start=1):
    subset = milestones.loc[milestones.support.eq(support) & milestones.day.notna()]
    for threshold in ("T50", "T80", "T90"):
        group = subset.loc[subset.threshold.eq(threshold)]
        fig.add_trace(go.Scatter(
            x=group.day, y=group.unit_name, mode="markers",
            marker=dict(size=10, color=threshold_colors[threshold], symbol=threshold_symbols[threshold]),
            name=threshold, legendgroup=threshold, showlegend=(column == 1),
            customdata=np.column_stack([group.status, group.quality_flag]),
            hovertemplate=(
                "%{y}<br>" + threshold + ": day %{x:.0f}"
                "<br>Status: %{customdata[0]}<br>Observability: %{customdata[1]}<extra></extra>"
            ),
        ), row=1, col=column)
    fig.update_xaxes(title_text="Persistent return day since Haiyan", row=1, col=column)

style_figure(fig, "How the trajectories become comparable recovery milestones")
fig.update_yaxes(categoryorder="array", categoryarray=STUDY_AREAS[::-1], row=1, col=1)
fig.show()

## Ordinary trajectories versus a functional boxplot

The ordinary plot retains every named trajectory and is the primary diagnostic. The functional boxplot answers a narrower question: what whole trajectory is most central, what envelope contains the deepest half of the curves, and which complete curves are unusually far from that central set?

It does not estimate a “better” recovery curve and does not replace the individual profiles. With only 12 study areas, it is an explanatory summary rather than a population-level result.

In [ ]:
# ============================================================
# 9. COMMON 20-DAY FEATURES AND FUNCTIONAL SUMMARY
# ============================================================

def build_feature_matrix(profiles, bin_days=20, horizon_days=180, min_count=2):
    selected = profiles.loc[
        profiles.block.between(0, (horizon_days // AGGREGATION_DAYS) - 1)
    ].copy()
    selected["cluster_bin"] = (
        selected.block.astype(int) * AGGREGATION_DAYS // bin_days
    ).astype(int)
    number_of_bins = horizon_days // bin_days
    grouped = selected.groupby(["unit_name", "cluster_bin"], observed=True).recovery_pct
    values = grouped.median().unstack("cluster_bin").reindex(columns=range(number_of_bins))
    counts = grouped.count().unstack("cluster_bin").reindex(index=values.index, columns=range(number_of_bins)).fillna(0)
    values = values.where(counts >= min_count)
    labels = [f"{index * bin_days}–{(index + 1) * bin_days - 1} d" for index in range(number_of_bins)]
    values.columns = labels
    counts.columns = labels
    return values, counts

def functional_summary(values):
    values = np.asarray(values, float)
    n = len(values)
    below = rankdata(values, method="min", axis=0) - 1
    above = n - rankdata(values, method="max", axis=0)
    depth = (1 - (below * (below - 1) + above * (above - 1)) / (n * (n - 1))).mean(axis=1)
    order = np.argsort(-depth, kind="stable")
    central = values[order[:int(np.ceil(n / 2))]]
    return {
        "depth": depth, "order": order, "median": values[order[0]],
        "low": central.min(axis=0), "high": central.max(axis=0),
        "outer_low": values.min(axis=0), "outer_high": values.max(axis=0),
    }

municipality_features, municipality_counts = build_feature_matrix(municipality_four_day)
complete_features = municipality_features.dropna()
complete_counts = municipality_counts.loc[complete_features.index]
feature_days = np.arange(complete_features.shape[1]) * CLUSTER_BIN_DAYS + (CLUSTER_BIN_DAYS - 1) / 2

fig = make_subplots(rows=1, cols=2, subplot_titles=("Named trajectories", "Functional summary"),
                    shared_yaxes=True, horizontal_spacing=0.08)
for name, row in complete_features.iterrows():
    fig.add_trace(go.Scatter(
        x=feature_days, y=row, mode="lines+markers",
        line=dict(color=AREA_COLORS.get(name, "#777777"), width=1.8),
        marker=dict(size=4), name=name,
        hovertemplate="%{fullData.name}<br>Day %{x:.0f}<br>Recovery %{y:.1f}%<extra></extra>",
    ), row=1, col=1)

summary = functional_summary(complete_features.values)
for low, high, color, label in [
    (summary["outer_low"], summary["outer_high"], "rgba(160,170,180,.22)", "All complete curves"),
    (summary["low"], summary["high"], "rgba(0,158,115,.32)", "Deepest 50%"),
]:
    fig.add_trace(go.Scatter(x=feature_days, y=low, mode="lines", line=dict(width=0),
                             showlegend=False, hoverinfo="skip"), row=1, col=2)
    fig.add_trace(go.Scatter(x=feature_days, y=high, mode="lines", line=dict(width=0),
                             fill="tonexty", fillcolor=color, name=label), row=1, col=2)

deepest_name = complete_features.index[summary["order"][0]]
fig.add_trace(go.Scatter(
    x=feature_days, y=summary["median"], mode="lines+markers",
    line=dict(color="#006644", width=4), marker=dict(size=6),
    name=f"Functional median: {deepest_name}",
), row=1, col=2)
for column in (1, 2):
    fig.add_hline(y=100, line_dash="dot", line_color=BASELINE_COLOR, row=1, col=column)
    fig.update_xaxes(title_text="Days since Haiyan", row=1, col=column)
style_figure(fig, "Individual recovery curves versus a functional boxplot")
fig.update_yaxes(title_text="Recovery (% baseline)", row=1, col=1)
fig.show()

In [ ]:
# ============================================================
# FIGURE 8. THE EXACT VALUES AND COUNTS USED FOR GROUPING
# ============================================================

fig = make_subplots(rows=1, cols=2, subplot_titles=("20-day median recovery", "Observed four-day composites"),
                    horizontal_spacing=0.12)
fig.add_trace(go.Heatmap(
    x=complete_features.columns, y=complete_features.index, z=complete_features.values,
    colorscale="Viridis", zmin=0, zmax=max(150, np.nanquantile(complete_features.values, 0.98)),
    colorbar=dict(title="Recovery %", x=0.46),
    hovertemplate="%{y}<br>%{x}<br>Recovery: %{z:.1f}%<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Heatmap(
    x=complete_counts.columns, y=complete_counts.index, z=complete_counts.values,
    colorscale="Blues", zmin=0, zmax=CLUSTER_BIN_DAYS // AGGREGATION_DAYS,
    colorbar=dict(title="Composites", x=1.01),
    hovertemplate="%{y}<br>%{x}<br>Observed composites: %{z:.0f}<extra></extra>",
), row=1, col=2)
style_figure(fig, "Before clustering: verify every trajectory feature and its evidence")
fig.update_yaxes(title_text="Study area", row=1, col=1)
fig.show()

## Group recovery behaviour only after inspecting the inputs

K-means below uses the nine 20-day recovery values. The values are not standardised within each area because persistent below-baseline or above-baseline levels are part of the behaviour being compared. The number of families is shown for several candidate values; the selected solution is a simple proof of concept, not a definitive typology.

For only 12 study areas, use the grouping to identify candidate analogues—for example, whether Tacloban is more similar to Ormoc or Palo—then return to the named curves, spatial maps and external evidence.

In [ ]:
# ============================================================
# FIGURE 9. CLUSTER-COUNT DIAGNOSTIC AND RECOVERY FAMILIES
# ============================================================

X = complete_features.to_numpy(float)
diagnostics = []
models = {}
for number_of_families in range(2, min(5, len(complete_features))):
    model = KMeans(n_clusters=number_of_families, n_init=100, random_state=CLUSTER_RANDOM_STATE).fit(X)
    diagnostics.append({
        "families": number_of_families,
        "silhouette": silhouette_score(X, model.labels_),
        "minimum_family_size": int(pd.Series(model.labels_).value_counts().min()),
    })
    models[number_of_families] = model

diagnostics = pd.DataFrame(diagnostics)
admissible = diagnostics.loc[diagnostics.minimum_family_size.ge(2)]
chosen_k = int((admissible if not admissible.empty else diagnostics).sort_values(
    ["silhouette", "minimum_family_size"], ascending=False
).iloc[0].families)
model = models[chosen_k]

centre_order = np.argsort(model.cluster_centers_.mean(axis=1))
label_map = {old_label: new_label for new_label, old_label in enumerate(centre_order)}
family_id = pd.Series([label_map[label] for label in model.labels_], index=complete_features.index, name="family_id")

fig = make_subplots(rows=1, cols=2, subplot_titles=("Candidate family counts", f"Selected {chosen_k}-family solution"),
                    horizontal_spacing=0.10)
fig.add_trace(go.Bar(
    x=diagnostics.families, y=diagnostics.silhouette,
    marker_color=np.where(diagnostics.families.eq(chosen_k), EVENT_COLOR, "#AAB4BE"),
    customdata=np.column_stack([diagnostics.minimum_family_size]),
    hovertemplate="Families: %{x}<br>Silhouette: %{y:.3f}<br>Smallest family: %{customdata[0]}<extra></extra>",
    showlegend=False,
), row=1, col=1)

for family in range(chosen_k):
    members = family_id.index[family_id.eq(family)]
    for name in members:
        fig.add_trace(go.Scatter(
            x=feature_days, y=complete_features.loc[name], mode="lines+markers",
            line=dict(color=FAMILY_COLORS[family], width=1.5), marker=dict(size=4), opacity=0.55,
            name=name, legendgroup=f"Family {family + 1}", showlegend=False,
            hovertemplate=f"{name}<br>Day %{{x:.0f}}<br>Recovery %{{y:.1f}}%<extra></extra>",
        ), row=1, col=2)
    centre = complete_features.loc[members].mean(axis=0)
    fig.add_trace(go.Scatter(
        x=feature_days, y=centre, mode="lines+markers",
        line=dict(color=FAMILY_COLORS[family], width=4, dash="dash"), marker=dict(size=7),
        name=f"Family {family + 1}: " + ", ".join(members),
        legendgroup=f"Family {family + 1}",
    ), row=1, col=2)

fig.add_hline(y=100, line_dash="dot", line_color=BASELINE_COLOR, row=1, col=2)
style_figure(fig, "From inspected trajectories to candidate recovery families", legend_y=1.12)
fig.update_xaxes(title_text="Number of families", dtick=1, row=1, col=1)
fig.update_yaxes(title_text="Silhouette score", row=1, col=1)
fig.update_xaxes(title_text="Days since Haiyan", row=1, col=2)
fig.update_yaxes(title_text="Recovery (% baseline)", row=1, col=2)
fig.show()

for family in range(chosen_k):
    print(f"Family {family + 1}:", ", ".join(family_id.index[family_id.eq(family)]))

In [ ]:
# ============================================================
# FIGURE 10. MAP THE TEMPORAL FAMILIES BACK TO SPACE
# ============================================================

map_gdf = municipalities_display.copy()
map_gdf["family_id"] = map_gdf.unit_name.map(family_id)
map_gdf["family"] = map_gdf.family_id.map(lambda value: f"Family {int(value) + 1}" if pd.notna(value) else "Not grouped")

fig = go.Figure()
for geometry in regional_boundaries.geometry.boundary:
    lines = geometry.geoms if geometry.geom_type == "MultiLineString" else [geometry]
    for line in lines:
        longitude, latitude = line.xy
        fig.add_trace(go.Scattergeo(
            lon=longitude, lat=latitude, mode="lines",
            line=dict(color="#C2C7CC", width=0.6), hoverinfo="skip", showlegend=False,
        ))

for family in range(chosen_k):
    subset = map_gdf.loc[map_gdf.family_id.eq(family)]
    for _, row in subset.iterrows():
        geometry = row.geometry.boundary
        lines = geometry.geoms if geometry.geom_type == "MultiLineString" else [geometry]
        for line in lines:
            longitude, latitude = line.xy
            fig.add_trace(go.Scattergeo(
                lon=longitude, lat=latitude, mode="lines",
                line=dict(color=FAMILY_COLORS[family], width=3),
                name=f"Family {family + 1}", legendgroup=f"Family {family + 1}",
                showlegend=False, hoverinfo="skip",
            ))
        point = row.geometry.representative_point()
        fig.add_trace(go.Scattergeo(
            lon=[point.x], lat=[point.y], mode="markers+text",
            marker=dict(size=10, color=FAMILY_COLORS[family], line=dict(color="white", width=1)),
            text=[row.unit_name], textposition="top center",
            name=f"Family {family + 1}", legendgroup=f"Family {family + 1}",
            showlegend=False,
            hovertemplate=f"{row.unit_name}<br>Family {family + 1}<extra></extra>",
        ))

fig.update_geos(fitbounds="locations", visible=False, projection_type="mercator")
fig.update_layout(
    template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", width=1280, height=720,
    title=dict(text="Where the candidate temporal recovery families occur", x=0.02, xanchor="left"),
    font=dict(family="Arial", size=14, color="#243B5A"),
    margin=dict(l=20, r=20, t=85, b=20),
)
fig.show()

## Check whether dim baselines are manufacturing volatile “recovery”

A small baseline can turn a modest absolute fluctuation into a large percentage change. The diagnostic below compares baseline brightness with pre-event variability. Areas that are both dim and highly variable should not be interpreted solely from normalised recovery percentages.

The test does not automatically remove the dimmest quartile. It identifies cases requiring review of raw radiance, pixel support, the 5×5 versus municipality-wide comparison, and external evidence.

In [ ]:
# ============================================================
# FIGURE 11. BASELINE BRIGHTNESS VERSUS PRE-EVENT VARIABILITY
# ============================================================

diagnostic_rows = []
for name, group in municipality_four_day.groupby("unit_name", observed=True):
    baseline = municipality_reports.set_index("unit_name").loc[name]
    pre_event = group.loc[
        group.date_start.between(BASELINE_START, PRE_EVENT_END), "recovery_pct"
    ].dropna()
    diagnostic_rows.append({
        "unit_name": name,
        "baseline_median_ntl": baseline.baseline_median_ntl,
        "fixed_baseline_pixels": baseline.fixed_baseline_pixels,
        "pre_event_mad_pct": float(np.median(np.abs(pre_event - pre_event.median()))) if len(pre_event) else np.nan,
        "family": f"Family {int(family_id[name]) + 1}" if name in family_id else "Not grouped",
    })
brightness_diagnostic = pd.DataFrame(diagnostic_rows)
brightness_cutoff = brightness_diagnostic.baseline_median_ntl.quantile(0.25)
brightness_diagnostic["baseline_group"] = np.where(
    brightness_diagnostic.baseline_median_ntl.lt(brightness_cutoff),
    "Dimmest quartile", "Brighter three quartiles",
)

fig = px.scatter(
    brightness_diagnostic,
    x="baseline_median_ntl", y="pre_event_mad_pct",
    size="fixed_baseline_pixels", color="baseline_group", text="unit_name",
    hover_data=["family", "fixed_baseline_pixels"],
    color_discrete_map={"Dimmest quartile": "#D55E00", "Brighter three quartiles": "#0072B2"},
    labels={
        "baseline_median_ntl": "Median pixel baseline radiance (nW cm⁻² sr⁻¹)",
        "pre_event_mad_pct": "Pre-event variability (MAD, percentage points)",
        "baseline_group": "Baseline brightness",
    },
)
fig.update_traces(textposition="top center")
fig.add_vline(x=brightness_cutoff, line_dash="dash", line_color="#D55E00")
style_figure(fig, "Is apparent recovery behaviour sensitive to a dim, volatile baseline?")
fig.show()

## Interpretation sequence before systematic expansion

For each study area:

1. **Observability:** verify the spatial support, retained four-day observations and missing intervals.
2. **Spatial change:** inspect whether the baseline-to-shock change is widespread or concentrated.
3. **Temporal change:** inspect the named municipality-wide and 5×5 trajectories without connecting gaps.
4. **Milestones:** interpret T50/T80/T90 only when the threshold was lost, the persistent return was observed and the quality flag is acceptable.
5. **Baseline sensitivity:** flag dim, highly variable supports and compare raw radiance with normalised recovery.
6. **Functional summary:** describe the central envelope of whole trajectories without replacing individual curves.
7. **Grouping:** identify candidate recovery analogues, then map them back to space.
8. **Explanation:** compare NTL families with physical EO, electricity, infrastructure, demographic and survey evidence.

The immediate output is a transparent set of candidate recovery families for these 12 locations. It is not yet a systematic classification of every municipality or pixel.

In [ ]:
# ============================================================
# 12. EXPORT ONLY THE CORE REPRODUCIBLE OUTPUTS
# ============================================================

municipality_four_day.to_csv(TABLE_DIR / "focused_municipality_four_day_profiles.csv", index=False)
kernel_four_day.to_csv(TABLE_DIR / "focused_5x5_four_day_profiles.csv", index=False)
municipality_metrics.to_csv(TABLE_DIR / "focused_municipality_metrics.csv", index=False)
kernel_metrics.to_csv(TABLE_DIR / "focused_5x5_metrics.csv", index=False)
kernel_anchors.to_csv(TABLE_DIR / "focused_5x5_anchors.csv", index=False)
complete_features.reset_index().to_csv(TABLE_DIR / "focused_clustering_features.csv", index=False)
family_id.rename("family_id").reset_index().to_csv(TABLE_DIR / "focused_recovery_families.csv", index=False)
brightness_diagnostic.to_csv(TABLE_DIR / "focused_baseline_variability.csv", index=False)

print("Core outputs:", TABLE_DIR)

## References

- Welle, T., et al. (2020). *A Proof-of-Concept of Integrating Machine Learning, Remote Sensing, and Survey Data in Evaluations: The Measurement of Disaster Resilience in the Philippines*. DEval Discussion Paper 1/2020.
- Román, M. O., et al. (2019). Satellite-based assessment of electricity restoration efforts in Puerto Rico after Hurricane Maria. *PLOS ONE, 14*(6), e0218883. https://doi.org/10.1371/journal.pone.0218883
- Zheng, Q., et al. (2025). Nighttime lights reveal substantial spatial heterogeneity and inequality in post-hurricane recovery. *Remote Sensing of Environment, 319*, 114645. https://doi.org/10.1016/j.rse.2025.114645
- Sun, Y., & Genton, M. G. (2011). Functional boxplots. *Journal of Computational and Graphical Statistics, 20*(2), 316–334. https://doi.org/10.1198/jcgs.2011.09224
- Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics, 20*, 53–65. https://doi.org/10.1016/0377-0427(87)90125-7